In [1]:
import os, numpy as np, pandas as pd

print(os.listdir("../input"))



['sample_submission.csv', 'test.csv.zip', 'train.csv.zip', 'description.md', 'jigsaw-toxic-comment-classification-challenge', 'sample_submission.csv.zip', 'train.csv', 'test.csv']


In [2]:
# tensorflow.keras imports (compatible with TF 2.x)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, GlobalMaxPool1D
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tqdm.auto import tqdm

tqdm.pandas()



2026-05-02 20:56:06.504536: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-02 20:56:06.514164: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777755366.525383      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777755366.528837      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 20:56:06.542102: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [3]:
# paths
DATA_PATH = "../input/jigsaw-toxic-comment-classification-challenge"
TRAIN_PATH = f"{DATA_PATH}/train.csv"
TEST_PATH = f"{DATA_PATH}/test.csv"
SAMPLE_SUB_PATH = f"{DATA_PATH}/sample_submission.csv"



In [4]:
df = pd.read_csv(TRAIN_PATH)
list_classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
y = df[list_classes].values



In [5]:
# Tokenization
MAX_WORDS = 20000
MAX_LEN = 50
token = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
token.fit_on_texts(df["comment_text"].astype(str))
seq = token.texts_to_sequences(df["comment_text"].astype(str))
pad_seq = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
vocab_size = min(MAX_WORDS, len(token.word_index) + 1)



In [6]:
# Random embedding matrix (no external GloVe file)
EMB_DIM = 50
embedding_matrix = np.random.normal(0, 0.1, (vocab_size, EMB_DIM)).astype(np.float32)



In [7]:
# Simple Bi‑LSTM model
model = Sequential()
model.add(
    Embedding(
        vocab_size,
        EMB_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=False,
    )
)
model.add(Bidirectional(LSTM(50, return_sequences=True)))
model.add(GlobalMaxPool1D())
model.add(Dense(50, activation="relu"))
model.add(Dropout(0.2))
model.add(Dense(6, activation="sigmoid"))
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()



/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
2026-05-02 20:56:24.705490: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │     1,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,000,000 (3.81 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 1,000,000 (3.81 MB)

In [8]:
# Train (small number of epochs for speed)
history = model.fit(
    pad_seq, y, epochs=2, batch_size=32, validation_split=0.1, verbose=2
)



Epoch 1/2
4488/4488 - 56s - 12ms/step - accuracy: 0.9531 - loss: 0.1109 - val_accuracy: 0.9940 - val_loss: 0.0864
Epoch 2/2
4488/4488 - 54s - 12ms/step - accuracy: 0.9942 - loss: 0.0799 - val_accuracy: 0.9940 - val_loss: 0.0768


In [9]:
# Prepare test data
test_df = pd.read_csv(TEST_PATH)
test_seq = token.texts_to_sequences(test_df["comment_text"].astype(str))
test_pad_seq = pad_sequences(
    test_seq, maxlen=MAX_LEN, padding="post", truncating="post"
)



In [10]:
# Predict
pred = model.predict(test_pad_seq, batch_size=256)



599/599 ━━━━━━━━━━━━━━━━━━━━ 9s 15ms/step


In [11]:
# Create submission
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
sample_submission[list_classes] = pred
submission_path = "submission.csv"
sample_submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")

Submission saved to submission.csv
